# BERT IMDb Text Classification with Hugging Face Transformers

Kaggle Notebook에서 바로 실행 가능한 Hugging Face Transformers 기반 BERT 텍스트 분류 예제입니다.

포함 내용:
1. `datasets` 라이브러리로 IMDb 데이터셋 로드
2. `bert-base-uncased` 모델 사용
3. `AutoTokenizer`로 텍스트 전처리
4. `AutoModelForSequenceClassification` 사용
5. `Trainer` API로 학습 수행
6. accuracy 평가
7. 예측 결과 출력

참고: Kaggle CPU 환경에서도 빠르게 실행되도록 IMDb 데이터셋 일부만 사용합니다. 더 오래 학습하려면 subset 크기와 epoch 수를 늘리면 됩니다.

In [ ]:
# Kaggle Notebook에서 필요한 라이브러리 설치
# transformers: BERT 모델과 Trainer API
# datasets: IMDb 데이터셋 로드
# accelerate: Trainer 실행에 필요
!pip install -q -U "transformers==4.48.3" datasets accelerate

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import numpy as np
import torch
import transformers
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 1. IMDb 데이터셋 로드

`datasets` 라이브러리의 IMDb 데이터셋을 사용합니다. IMDb는 영화 리뷰 텍스트를 긍정/부정으로 분류하는 이진 분류 데이터셋입니다.

In [ ]:
# IMDb 데이터셋을 다운로드하고 로드합니다.
raw_datasets = load_dataset("imdb")

# Kaggle에서 빠르게 실행되도록 일부 샘플만 사용합니다.
# 실제 학습 성능을 높이려면 train/test 샘플 수를 늘리면 됩니다.
train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(800))
eval_dataset = raw_datasets["test"].shuffle(seed=42).select(range(200))

print(train_dataset)
print(eval_dataset)
print("\nSample text:")
print(train_dataset[0]["text"][:500])
print("Label:", train_dataset[0]["label"])

## 2. AutoTokenizer로 전처리

`bert-base-uncased` 토크나이저를 사용해 IMDb 리뷰 텍스트를 BERT 입력 형식으로 변환합니다.

In [ ]:
# BERT 기본 영어 uncased 모델의 토크나이저를 불러옵니다.
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 텍스트를 토큰 ID로 변환하는 전처리 함수입니다.
# truncation=True로 BERT 최대 입력 길이를 넘는 문장을 잘라냅니다.
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

# map을 사용해 train/eval 데이터셋 전체에 토크나이징을 적용합니다.
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# DataCollatorWithPadding은 배치마다 가장 긴 문장 길이에 맞춰 동적으로 padding합니다.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_train[0].keys())

## 3. AutoModelForSequenceClassification 로드

`bert-base-uncased`를 이진 분류용 모델로 불러옵니다. IMDb label은 `0: negative`, `1: positive`입니다.

In [ ]:
# IMDb는 부정/긍정 이진 분류이므로 num_labels=2로 설정합니다.
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

## 4. Accuracy 평가 함수 정의

`Trainer`가 평가할 때 사용할 accuracy 계산 함수를 정의합니다.

In [ ]:
# eval_pred는 모델 출력 logits와 정답 labels를 포함합니다.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = np.mean(predictions == labels)
    return {"accuracy": accuracy}

## 5. Trainer API로 학습

`TrainingArguments`와 `Trainer`를 설정한 뒤 BERT 모델을 IMDb subset으로 fine-tuning합니다.

In [ ]:
# Kaggle에서 빠르게 실행되도록 epoch과 batch size를 작게 설정합니다.
training_args = TrainingArguments(
    output_dir="./bert-imdb-results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
)

# Trainer는 학습 루프, 평가, 배치 생성 등을 자동으로 처리합니다.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 모델 학습을 수행합니다.
trainer.train()

## 6. 평가 Accuracy 확인

학습된 모델을 평가 데이터셋으로 평가하고 accuracy를 출력합니다.

In [ ]:
# 평가 데이터셋에 대한 loss와 accuracy를 계산합니다.
eval_results = trainer.evaluate()

print("Evaluation results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

## 7. 예측 결과 출력

학습된 BERT 모델로 새로운 영화 리뷰 문장을 분류합니다.

In [ ]:
# 예측할 새 IMDb 스타일 리뷰 문장입니다.
test_texts = [
    "This movie was fantastic. The acting was great and the story was deeply moving.",
    "I regret watching this film. It was boring, slow, and badly written.",
    "The movie had some good scenes, but overall it felt too predictable.",
]

# tokenizer로 새 문장을 모델 입력 형태로 변환합니다.
inputs = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt",
)

# 모델이 올라간 device와 입력 tensor device를 맞춥니다.
device = trainer.model.device
inputs = {key: value.to(device) for key, value in inputs.items()}

# 예측 시에는 gradient 계산이 필요 없으므로 torch.no_grad()를 사용합니다.
trainer.model.eval()
with torch.no_grad():
    outputs = trainer.model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_labels = torch.argmax(probabilities, dim=-1)

# 예측 label과 confidence를 출력합니다.
for text, label_id, probs in zip(test_texts, predicted_labels, probabilities):
    label_id = label_id.item()
    confidence = probs[label_id].item()
    print("Text:", text)
    print("Prediction:", id2label[label_id])
    print("Confidence:", round(confidence, 4))
    print("-" * 100)